# 02 — Data preprocessing


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import json

# Project root
BASE = Path("..")

# Dataset path
# DATA_PATH = BASE / "data" / "raw" / "patient_medicine_reviews.csv"
# DATA_PATH = BASE / "data" / "raw" / "patient_medicine_reviews_20000_synthetic.csv"
# DATA_PATH = BASE / "data" / "raw" / "patient_medicine_reviews_enhanced.csv"
DATA_PATH = BASE / "data" / "raw" / "medireview_comprehensive_60000.csv"
# Load dataset
df = pd.read_csv(DATA_PATH)

df.head()

,id,gender,age,medicine,condition,review,label
0,0,Female,62.0,Gabapentin,Nerve Pain,<br>I filled the Gabapentin prescription for n...,no side effect
1,1,Other,55.0,Azithromycin,Bacterial Infection,I was given Azithromycin to treat bacterial in...,side effect
2,2,Female,57.0,Sitagliptin,Type 2 Diabetes,Sitagliptin for type 2 diabetes. I expected ac...,no side effect
3,3,Female,NaN,Cetirizine,Allergic Rhinitis,Checking in after three days with Cetirizine. ...,neutral
4,4,Other,71.0,Prednisone,Inflammation,Prednisone for inflammation. I was asked to co...,side effect


In [2]:
df["age"] = pd.to_numeric(df["age"], errors="coerce")

In [3]:
# Step 1: mask for out-of-range ages
invalid_age = (df["age"] > 88) | (df["age"] < 12)
print(invalid_age.sum())  

# Step 2: set only those to missing
df.loc[invalid_age, "age"] = pd.NA

# Step 3: confirm it worked
print(((df["age"] > 88) | (df["age"] < 12)).sum())   # 0
print(df["age"].isnull().sum())

0
0
514


In [4]:
df = df.dropna(subset=['age'])

In [5]:
df['age'].isnull().sum()

np.int64(0)

In [6]:
df.duplicated().sum()

np.int64(0)

In [7]:
df['gender'].unique()

<ArrowStringArray>
['Female', 'Other', 'Male', nan]
Length: 4, dtype: str

In [8]:
df=df.dropna(subset=['gender'])

In [9]:
df['gender'].isnull().sum()

np.int64(0)

In [10]:
df.isnull().sum()

id             0
gender         0
age            0
medicine     168
condition    329
review        96
label         96
dtype: int64

In [11]:
df.dropna(inplace=True)

In [12]:
df["review_no_html"] = df["review"].str.replace(r"<[^>]+>", " ", regex=True)

In [13]:
import re

emoji_pattern = re.compile(
    r'[\U0001F300-\U0001FAFF\u2600-\u27BF]'
)

emojis = set()

for text in df['review'].dropna():
    emojis.update(emoji_pattern.findall(text))

print(emojis)

{'👎', '👍', '😊', '😡', '😞', '🤮', '💊'}


In [14]:
emoji_map = {
    "🤮": "vomit",
    "👎": "thumbsdown",
    "😡": "angry",
    "😞": "sad",
    "👍": "thumbsup",
    "💊": "pill",
    "😊": "happy"
}

df["patient_review"] = df["review_no_html"].replace(emoji_map, regex=True)

In [15]:
df.head(10)

,id,gender,age,medicine,condition,review,label,review_no_html,patient_review
0,0,Female,62.0,Gabapentin,Nerve Pain,<br>I filled the Gabapentin prescription for n...,no side effect,I filled the Gabapentin prescription for nerv...,I filled the Gabapentin prescription for nerv...
1,1,Other,55.0,Azithromycin,Bacterial Infection,I was given Azithromycin to treat bacterial in...,side effect,I was given Azithromycin to treat bacterial in...,I was given Azithromycin to treat bacterial in...
2,2,Female,57.0,Sitagliptin,Type 2 Diabetes,Sitagliptin for type 2 diabetes. I expected ac...,no side effect,Sitagliptin for type 2 diabetes. I expected ac...,Sitagliptin for type 2 diabetes. I expected ac...
4,4,Other,71.0,Prednisone,Inflammation,Prednisone for inflammation. I was asked to co...,side effect,Prednisone for inflammation. I was asked to co...,Prednisone for inflammation. I was asked to co...
6,6,Female,62.0,Sertraline,Depression,<br>The treatment I mean throughout this revie...,side effect,The treatment I mean throughout this review i...,The treatment I mean throughout this review i...
7,7,Male,82.0,Omeprazole,GERD,three weeks into treatment with Omeprazole: I ...,neutral,three weeks into treatment with Omeprazole: I ...,three weeks into treatment with Omeprazole: I ...
8,8,Other,71.0,Topiramate,Migraine Prevention,here is my update on topiramate after a fortni...,neutral,here is my update on topiramate after a fortni...,here is my update on topiramate after a fortni...
9,9,Female,62.0,Gabapentin,Nerve Pain,here is my update on gabapentin after six days...,no side effect,here is my update on gabapentin after six days...,here is my update on gabapentin after six days...
10,10,Male,54.0,Levothyroxine,Hypothyroidism,Taking Levothyroxine; the reason is hypothyroi...,no side effect,Taking Levothyroxine; the reason is hypothyroi...,Taking Levothyroxine; the reason is hypothyroi...
11,11,Male,53.0,Cetirizine,Allergic Rhinitis,<p>I filled the Cetirizine prescription for al...,neutral,I filled the Cetirizine prescription for alle...,I filled the Cetirizine prescription for alle...


In [16]:
df['age']=df['age'].astype('int')

In [17]:
df=df.drop(columns=['review','review_no_html'])

In [18]:
df

,id,gender,age,medicine,condition,label,patient_review
0,0,Female,62,Gabapentin,Nerve Pain,no side effect,I filled the Gabapentin prescription for nerv...
1,1,Other,55,Azithromycin,Bacterial Infection,side effect,I was given Azithromycin to treat bacterial in...
2,2,Female,57,Sitagliptin,Type 2 Diabetes,no side effect,Sitagliptin for type 2 diabetes. I expected ac...
4,4,Other,71,Prednisone,Inflammation,side effect,Prednisone for inflammation. I was asked to co...
6,6,Female,62,Sertraline,Depression,side effect,The treatment I mean throughout this review i...
...,...,...,...,...,...,...,...
15995,10807,Female,18,Sumatriptan,Migraine,neutral,I am writing about my current treatment. Nothi...
15996,11913,Female,74,Allopurinol,Gout,neutral,My review is about Allopurinol. I have not com...
15997,11542,Female,85,Pantoprazole,Acid Reflux,neutral,I would leave the effectiveness rating blank f...
15998,12660,Male,76,Losartan,Hypertension,neutral,I started Losartan recently. I have my follow-...


In [19]:
df['patient_review']=df['patient_review'].apply(lambda x: x.lower())

In [20]:
# Sort label names so the mapping does not depend on row order.
df["label"] = df["label"].astype(str).str.strip()
label_dict = {label: i for i, label in enumerate(sorted(df["label"].unique()))}
print(label_dict)
df["label"] = df["label"].map(label_dict)

{'neutral': 0, 'no side effect': 1, 'side effect': 2}


In [21]:
df=df[['id','gender','age','condition','patient_review','label']]

In [22]:
df["patient_review"] = (
    df["patient_review"]
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

In [ ]:
import re


def expand_negations(text):

    text = str(text)

    text = text.replace("’", "'")

    # Special contractions
    text = re.sub(
        r"\bwon't\b",
        "will not",
        text,
        flags=re.IGNORECASE
    )

    text = re.sub(
        r"\bcan't\b",
        "can not",
        text,
        flags=re.IGNORECASE
    )

    text = re.sub(
        r"\bshan't\b",
        "shall not",
        text,
        flags=re.IGNORECASE
    )

    # didn't -> did not
    # wasn't -> was not
    # isn't -> is not
    # couldn't -> could not
    text = re.sub(
        r"\b(\w+)n't\b",
        r"\1 not",
        text,
        flags=re.IGNORECASE
    )

    return text


df["patient_review"] = (
    df["patient_review"]
    .apply(expand_negations)
)

In [23]:
import json

# 1. Remove empty reviews
df = df[df["patient_review"].str.len() > 0]

# 2. Remove duplicate reviews
df = df.drop_duplicates(subset=["patient_review"])

# 3. Save cleaned dataset
df.to_csv("../data/processed/cleaned_reviews.csv", index=False)

# 4. Save label mapping
with open("../artifacts/label_mapping.json", "w") as file:
    json.dump(label_dict, file, indent=2)

print("Saved cleaned rows:", len(df))

df.head()

Saved cleaned rows: 14462


,id,gender,age,condition,patient_review,label
0,0,Female,62,Nerve Pain,i filled the gabapentin prescription for nerve...,1
1,1,Other,55,Bacterial Infection,i was given azithromycin to treat bacterial in...,2
2,2,Female,57,Type 2 Diabetes,sitagliptin for type 2 diabetes. i expected ac...,1
4,4,Other,71,Inflammation,prednisone for inflammation. i was asked to co...,2
6,6,Female,62,Depression,the treatment i mean throughout this review is...,2
